# Phase 4b addendum: every-step readouts and weight distances (A100 80 GB)

Reads `results/phase4b/supp100_runs.csv`, `results/phase4b/distances_nearest.csv` (written by `python -m zoo.phase4b_followup`), the every-step curves in `results/colab/queue_4b_followup/runs/` and the long distance table there. Trains nothing. Writes `reports/figures/phase4b_addendum.png`.

Panel A is supplementary: the verdicts stay those of the frozen 10-step grid.

In [ ]:
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
F = ROOT / "results/colab/queue_4b_followup/runs"
R = pd.read_csv(ROOT / "results/phase4b/supp100_runs.csv")
D = pd.read_csv(F / "a100dist4bdistances/distances.csv")
R[R.arm != "control"].groupby("arm").t_cross_every_step.describe()

In [ ]:
INK, MUTED, GRID = "#222222", "#6b6b6b", "#e6e6e6"
C = {"student": "#c0504d", "1x-teacher student": "#d08a1f", "sham student": "#4f6d8f"}
LAB = {"student": "active-teacher students", "1x-teacher student": "1x-teacher students", "sham student": "sham students"}
fig, (A, B) = plt.subplots(1, 2, figsize=(11.5, 4.4), constrained_layout=True)
for r in R[R.arm != "control"].itertuples():
    c = pd.read_csv(F / r.run_id / "curve.csv").dropna(subset=["target_logprob"])
    A.plot(c.step, c.target_logprob, color=C[r.arm], lw=1, alpha=0.55)
    if pd.notna(r.t_cross_every_step):
        A.plot([r.t_cross_every_step], [r.logprob_at_cross], "o", ms=4, color=C[r.arm], mec="white", mew=0.5, zorder=3)
A.axhline(-1, color=INK, lw=0.8, ls="--"); A.text(99, -1.9, "threshold -1", va="center", ha="right", fontsize=8, color=INK)
for x in range(0, 101, 10):
    A.axvline(x, color=GRID, lw=0.6, zorder=0)
for arm, (x, y) in {"student": (13, -3.4), "1x-teacher student": (15, -12.8), "sham student": (58, -7.2)}.items():
    A.text(x, y, LAB[arm], color=C[arm], fontsize=8.5)
A.set_xlim(0, 101); A.set_ylim(-18, 1); A.set_xlabel("assay step (every step; grey lines = the frozen 10-step grid)")
A.set_ylabel("target log-prob, triggered prompts")
A.set_title("A  First 100 assay steps, 64x students, 5 seeds x 3 assay seeds", loc="left", fontsize=10)
A.text(0.99, 0.02, "dots = every-step crossings\nsham students start at about -65 (off scale)\nand cross at 580-760", transform=A.transAxes,
       ha="right", va="bottom", fontsize=7.5, color=MUTED)

T = D[D.kind == "student to teacher"].copy()
T["sseed"] = T.model.str.extract(r"-s(\d)-d64x")[0].astype(int); T["tseed"] = T.reference.str.extract(r"-s(\d)")[0].astype(int)
T["tkind"] = T.reference.map(lambda s: "naive" if "naive" in s else "active or 1x")
T["arm"] = T.model.map(lambda m: "1x-teacher student" if "4b1x" in m else ("sham student" if "sham" in m else "student"))
own_kind = {"student": "active or 1x", "1x-teacher student": "active or 1x", "sham student": "naive"}
base = D[D.kind == "student to base"].set_index("model").matrix_norm
tbase = D[D.kind == "teacher to base"].matrix_norm
cats = ["own teacher", "other same-seed teacher", "other seeds' teachers", "base model"]
jit = {"student": -0.2, "1x-teacher student": 0.0, "sham student": 0.2}
for arm in C:
    t = T[T.arm == arm]
    same = t[t.sseed == t.tseed]
    vals = [same[same.tkind == own_kind[arm]].groupby("model").matrix_norm.min(),
            same[same.tkind != own_kind[arm]].groupby("model").matrix_norm.min(),
            t[t.sseed != t.tseed].groupby("model").matrix_norm.min(),
            base[base.index.isin(t.model)]]
    for i, v in enumerate(vals):
        B.scatter(np.full(len(v), i + jit[arm]), v.values, s=22, color=C[arm], edgecolor="white", linewidth=0.5, zorder=3,
                  label=LAB[arm] if i == 0 else None)
B.axhspan(tbase.min(), tbase.max(), color=GRID, zorder=0)
B.text(1.5, tbase.mean() + 0.25, "band: teachers to base model", fontsize=7.5, color=MUTED, va="bottom", ha="center")
B.set_xticks(range(4), cats, rotation=15, ha="right"); B.set_ylim(0, 12.5)
B.set_ylabel("distance in the 48 layer matrices")
B.set_title("B  Each 64x student's distance to ...", loc="left", fontsize=10)
B.legend(frameon=False, fontsize=8, loc="center left", bbox_to_anchor=(0.0, 0.45))
B.text(0.99, 0.02, "nearest of each kind; fp16 rounding floor 0.012", transform=B.transAxes, ha="right", va="bottom",
       fontsize=7.5, color=MUTED)
for ax in (A, B):
    ax.spines[["top", "right"]].set_visible(False)
fig.savefig(ROOT / "reports/figures/phase4b_addendum.png", dpi=160)